In [ ]:
# 구글드라이브 연동 그리고 깃허브 클론/풀
import os
from google.colab import drive

drive.mount('/content/drive')

%cd /content
if os.path.exists('/content/korean-chatbot'):
    %cd korean-chatbot
    !git pull
else:
    !git clone https://github.com/kkkk2058/korean-chatbot.git
    %cd korean-chatbot

!pip install -r requirements.txt

In [ ]:
# 허깅페이스에서 데이터 가져오기
from datasets import load_dataset

# streaming=True 필수!
ds = load_dataset("heegyu/namuwiki-extracted", split="train", streaming=True)

In [ ]:
import re
import os
from datasets import load_dataset

ds = load_dataset("heegyu/namuwiki-extracted", split="train", streaming=True)

MAX_SAMPLES = 100_000

def clean(text):
    # ① [수정] 대괄호는 지우고 알맹이 글자만 남기기 (예: [[조선]] -> 조선)
    text = re.sub(r'\[\[(?:[^\]|]*\|)?([^\]]+)\]\]', r'\1', text)
    
    # ② URL 제거
    text = re.sub(r'https?://\S+', '', text)
    
    # ③ ReDoS 방지를 위해 안전한 패턴으로 문단 구분선(== 제목 ==) 제거
    text = re.sub(r'==[^=\n]+==', '', text)
    
    # ④ 연속된 공백을 하나로 축소
    text = re.sub(r'\s+', ' ', text)
    
    # ⑤ 만에 하나 짝이 안 맞아 남은 유령 대괄호('[', ']') 최종 청소
    text = text.replace('[', '').replace(']', '')
    
    return text.strip()

os.makedirs('data', exist_ok=True)

print("데이터 추출 및 정제 시작...")

buffer = []
saved_count = 0

with open('data/namuwiki.txt', 'w', encoding='utf-8') as f:
    for i, row in enumerate(ds):
        if saved_count >= MAX_SAMPLES:
            break
            
        text = clean(row['text'])
        
        if len(text) > 10:
            buffer.append(text + '\n')
            saved_count += 1
            
        # 1,000개씩 모아서 디스크에 한 번에 쓰기 (속도 향상)
        if len(buffer) >= 1000:
            f.writelines(buffer)
            buffer = []
            print(f"현재 {saved_count}개 저장 완료...")

    # 남은 버퍼 비우기
    if buffer:
        f.writelines(buffer)

print(f"완료! 총 {saved_count}개의 데이터가 저장되었습니다.")

In [ ]:
import shutil
import os

# 폴더 먼저 만들고
os.makedirs('/content/drive/MyDrive/korean-chatbot/data', exist_ok=True)

# 그 다음 복사
shutil.copy('data/namuwiki.txt', '/content/drive/MyDrive/korean-chatbot/data/namuwiki.txt')
print("구글 드라이브 저장 완료!")